In [26]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

In [27]:
df = pd.read_csv('dataMT.csv')
smiles = df['Smiles']
smiles


0         CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1
1          CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1
2       CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...
3            CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2
4       COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...
                              ...                        
4199                            O=C(O)/C=C/c1ccc(O)c(O)c1
4200    COc1ccc2c(c1)c(CC(=O)OCC(=O)O)c(C)n2C(=O)c1ccc...
4201    COc1ccc(/C=C/c2c(CC=C(C)C)c(O)cc(O)c2CC=C(C)C)...
4202                                 Nc1ccc(O)c(C(=O)O)c1
4203                         Cc1c(Cl)cccc1Nc1ccccc1C(=O)O
Name: Smiles, Length: 4204, dtype: object

Для расчета дескрипторов возмем список всех дескрипторов из RDKit

In [28]:
all_desc = Descriptors.descList
desc_names = [name for name, func in all_desc]
desc_names

['MaxAbsEStateIndex',
 'MaxEStateIndex',
 'MinAbsEStateIndex',
 'MinEStateIndex',
 'qed',
 'SPS',
 'MolWt',
 'HeavyAtomMolWt',
 'ExactMolWt',
 'NumValenceElectrons',
 'NumRadicalElectrons',
 'MaxPartialCharge',
 'MinPartialCharge',
 'MaxAbsPartialCharge',
 'MinAbsPartialCharge',
 'FpDensityMorgan1',
 'FpDensityMorgan2',
 'FpDensityMorgan3',
 'BCUT2D_MWHI',
 'BCUT2D_MWLOW',
 'BCUT2D_CHGHI',
 'BCUT2D_CHGLO',
 'BCUT2D_LOGPHI',
 'BCUT2D_LOGPLOW',
 'BCUT2D_MRHI',
 'BCUT2D_MRLOW',
 'AvgIpc',
 'BalabanJ',
 'BertzCT',
 'Chi0',
 'Chi0n',
 'Chi0v',
 'Chi1',
 'Chi1n',
 'Chi1v',
 'Chi2n',
 'Chi2v',
 'Chi3n',
 'Chi3v',
 'Chi4n',
 'Chi4v',
 'HallKierAlpha',
 'Ipc',
 'Kappa1',
 'Kappa2',
 'Kappa3',
 'LabuteASA',
 'PEOE_VSA1',
 'PEOE_VSA10',
 'PEOE_VSA11',
 'PEOE_VSA12',
 'PEOE_VSA13',
 'PEOE_VSA14',
 'PEOE_VSA2',
 'PEOE_VSA3',
 'PEOE_VSA4',
 'PEOE_VSA5',
 'PEOE_VSA6',
 'PEOE_VSA7',
 'PEOE_VSA8',
 'PEOE_VSA9',
 'SMR_VSA1',
 'SMR_VSA10',
 'SMR_VSA2',
 'SMR_VSA3',
 'SMR_VSA4',
 'SMR_VSA5',
 'SMR_VSA6',


In [29]:
def compute_descriptors(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        # Если SMILES невалидный, возвращаем None для всех дескрипторов
        return {name: None for name in desc_names}
    results = {}
    for name, func in all_desc:
        try:
            results[name] = func(mol)
        except Exception:
            results[name] = None
    return results

In [30]:
# Применяем к каждому SMILES и расширяем DataFrame
desc_df = smiles.apply(compute_descriptors).apply(pd.Series)
desc_df = pd.concat([smiles, desc_df], axis=1)
desc_df

,Smiles,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,12.372552,12.372552,0.037493,-3.264341,0.765113,21.320000,364.463,340.271,364.134445,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,11.641862,11.641862,0.279401,-3.231616,0.679285,11.480000,372.877,355.741,372.069926,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,13.229344,13.229344,0.097334,-0.303369,0.394970,10.896552,413.901,389.709,413.139386,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2,12.675413,12.675413,0.024250,-0.029042,0.654809,16.347826,316.485,284.229,316.240230,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...,13.438530,13.438530,0.025891,-0.497540,0.157660,11.048780,574.077,541.821,573.203049,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4199,O=C(O)/C=C/c1ccc(O)c(O)c1,10.125799,10.125799,0.229190,-1.062440,0.471621,10.461538,180.159,172.095,180.042259,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4200,COc1ccc2c(c1)c(CC(=O)OCC(=O)O)c(C)n2C(=O)c1ccc...,13.155765,13.155765,0.176906,-1.238810,0.619257,10.724138,415.829,397.685,415.082265,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4201,COc1ccc(/C=C/c2c(CC=C(C)C)c(O)cc(O)c2CC=C(C)C)...,10.501451,10.501451,0.058273,0.058273,0.397823,10.793103,394.511,364.271,394.214409,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4202,Nc1ccc(O)c(C(=O)O)c1,10.358426,10.358426,0.175926,-1.185370,0.408716,9.454545,153.137,146.081,153.042593,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Теперь у нас есть все данные о дескрипторах молекул. Проведем фильтрацию и удалим ненужные моменты.  

Удаляем дескрипторы с высокой корреляцией, потому что у них высокая зависимость: информация повторяется и мы избавляемся от лишнй размерности. 

In [31]:
# Матрица корреляций
corr_matrix = desc_df.copy()
corr_matrix = corr_matrix.drop(columns=['Smiles'])
corr_matrix = corr_matrix.corr().abs()

# Верхний треугольник матрицы (без диагонали)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Находим столбцы, у которых есть корреляция выше порога
threshold = 0.7
to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

# print("Будут удалены дескрипторы:", to_drop)

desc_df = desc_df.drop(columns=to_drop).reset_index(drop=True)
desc_df

,Smiles,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,NumRadicalElectrons,MaxPartialCharge,MinPartialCharge,...,fr_quatN,fr_sulfide,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,12.372552,0.037493,-3.264341,0.765113,21.320000,364.463,0.0,0.374465,-0.482924,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,11.641862,0.279401,-3.231616,0.679285,11.480000,372.877,0.0,0.175019,-0.260619,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,13.229344,0.097334,-0.303369,0.394970,10.896552,413.901,0.0,0.309845,-0.496743,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2,12.675413,0.024250,-0.029042,0.654809,16.347826,316.485,0.0,0.162398,-0.492062,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...,13.438530,0.025891,-0.497540,0.157660,11.048780,574.077,0.0,0.330326,-0.496743,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4199,O=C(O)/C=C/c1ccc(O)c(O)c1,10.125799,0.229190,-1.062440,0.471621,10.461538,180.159,0.0,0.327821,-0.504260,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4200,COc1ccc2c(c1)c(CC(=O)OCC(=O)O)c(C)n2C(=O)c1ccc...,13.155765,0.176906,-1.238810,0.619257,10.724138,415.829,0.0,0.341344,-0.496743,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4201,COc1ccc(/C=C/c2c(CC=C(C)C)c(O)cc(O)c2CC=C(C)C)...,10.501451,0.058273,0.058273,0.397823,10.793103,394.511,0.0,0.160018,-0.507497,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4202,Nc1ccc(O)c(C(=O)O)c1,10.358426,0.175926,-1.185370,0.408716,9.454545,153.137,0.0,0.339048,-0.507050,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Так же удалим где везде нули и NaN

In [34]:
cols_zero_or_nan = desc_df.fillna(0).columns[(desc_df.fillna(0) == 0).all()]
print('Пустые колонки:', list(cols_zero_or_nan))
desc_df = desc_df.drop(columns=cols_zero_or_nan).reset_index(drop=True)
desc_df

Пустые колонки: ['NumRadicalElectrons', 'SMR_VSA8', 'SlogP_VSA9', 'fr_benzodiazepine', 'fr_diazo', 'fr_dihydropyridine', 'fr_isocyan', 'fr_isothiocyan', 'fr_lactam', 'fr_phos_acid', 'fr_phos_ester', 'fr_prisulfonamd', 'fr_quatN']


,Smiles,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,MaxPartialCharge,MinPartialCharge,FpDensityMorgan1,...,fr_pyridine,fr_sulfide,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,12.372552,0.037493,-3.264341,0.765113,21.320000,364.463,0.374465,-0.482924,1.080000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,11.641862,0.279401,-3.231616,0.679285,11.480000,372.877,0.175019,-0.260619,0.960000,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,13.229344,0.097334,-0.303369,0.394970,10.896552,413.901,0.309845,-0.496743,1.103448,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2,12.675413,0.024250,-0.029042,0.654809,16.347826,316.485,0.162398,-0.492062,1.260870,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...,13.438530,0.025891,-0.497540,0.157660,11.048780,574.077,0.330326,-0.496743,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4199,O=C(O)/C=C/c1ccc(O)c(O)c1,10.125799,0.229190,-1.062440,0.471621,10.461538,180.159,0.327821,-0.504260,1.230769,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4200,COc1ccc2c(c1)c(CC(=O)OCC(=O)O)c(C)n2C(=O)c1ccc...,13.155765,0.176906,-1.238810,0.619257,10.724138,415.829,0.341344,-0.496743,1.103448,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4201,COc1ccc(/C=C/c2c(CC=C(C)C)c(O)cc(O)c2CC=C(C)C)...,10.501451,0.058273,0.058273,0.397823,10.793103,394.511,0.160018,-0.507497,0.793103,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4202,Nc1ccc(O)c(C(=O)O)c1,10.358426,0.175926,-1.185370,0.408716,9.454545,153.137,0.339048,-0.507050,1.454545,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
